In [1]:
import pandas as pd
import os

# Path to raw data folder
data_path = "../data/raw/"

# List of all CSV files
csv_files = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

# Load each file and print info
dataframes = {}
for file in csv_files:
    name = file.replace(".csv", "")
    df = pd.read_csv(data_path + file)
    dataframes[name] = df
    print(f"\n{'='*50}")
    print(f"File: {file}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))



File: 01_fund_master.csv
Shape: (40, 15)
Columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
   amfi_code       fund_house                                scheme_name  \
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund   SBI Bluechip Fund - Direct Plan - Growth   

  category sub_category     plan launch_date      benchmark  \
0   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01  NIFTY 100 TRI   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \
0               1.54            1.0             500                1000   
1               0.66            1.0             500                1000   

    fund_manager risk_category sebi_category_code  
0  Sohini An

In [2]:
# Clean NAV History

nav = dataframes["02_nav_history"]

# Date ko datetime me convert karo
nav["date"] = pd.to_datetime(nav["date"])

# AMFI code aur date ke hisab se sort karo
nav = nav.sort_values(["amfi_code", "date"])

# Missing NAV forward fill
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

# Duplicate rows hatao
nav = nav.drop_duplicates()

# NAV 0 se bada hona chahiye
nav = nav[nav["nav"] > 0]

print(nav.head())
print(nav.shape)

NameError: name 'dataframes' is not defined

In [3]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()
nav = nav.drop_duplicates()
nav = nav[nav["nav"] > 0]

print(nav.head())
print(nav.shape)

      amfi_code       date       nav
5750     100016 2022-01-03  520.4608
5751     100016 2022-01-04  515.0971
5752     100016 2022-01-05  521.7239
5753     100016 2022-01-06  515.7880
5754     100016 2022-01-07  515.1639
(46000, 3)


In [2]:
import requests
import json

# Fetch live NAV for HDFC Top 100 Direct (code: 125497)
url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)
data = response.json()

# Print fund info
print("Fund Name:", data['meta']['scheme_name'])
print("Fund House:", data['meta']['fund_house'])
print("Latest NAV Date:", data['data'][0]['date'])
print("Latest NAV:", data['data'][0]['nav'])

# Save as CSV
import pandas as pd
nav_df = pd.DataFrame(data['data'])
nav_df.to_csv("../data/raw/live_nav_125497.csv", index=False)
print("\nSaved to data/raw/live_nav_125497.csv")
print(nav_df.head())

Fund Name: SBI Small Cap Fund - Direct Plan - Growth
Fund House: SBI Mutual Fund
Latest NAV Date: 19-06-2026
Latest NAV: 202.07610

Saved to data/raw/live_nav_125497.csv
         date        nav
0  19-06-2026  202.07610
1  18-06-2026  200.95650
2  17-06-2026  199.83020
3  16-06-2026  198.61520
4  15-06-2026  198.03200


In [3]:
# Fetch NAV for 5 key schemes
schemes = {
    119551: "SBI Bluechip",
    120503: "ICICI Bluechip", 
    118632: "Nippon Large Cap",
    119092: "Axis Bluechip",
    120841: "Kotak Bluechip"
}

all_nav_data = []
for code, name in schemes.items():
    url = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    data = response.json()
    latest = data['data'][0]
    print(f"{name}: NAV = {latest['nav']} | Date = {latest['date']}")
    all_nav_data.append({
        'scheme_code': code,
        'scheme_name': name,
        'latest_nav': latest['nav'],
        'date': latest['date']
    })

# Save combined
nav_5_df = pd.DataFrame(all_nav_data)
nav_5_df.to_csv("../data/raw/live_nav_5schemes.csv", index=False)
print("\nSaved! Summary:")
print(nav_5_df)

SBI Bluechip: NAV = 105.92190 | Date = 19-06-2026
ICICI Bluechip: NAV = 107.96360 | Date = 19-06-2026
Nippon Large Cap: NAV = 100.78240 | Date = 19-06-2026
Axis Bluechip: NAV = 6195.78150 | Date = 19-06-2026
Kotak Bluechip: NAV = 251.57130 | Date = 19-06-2026

Saved! Summary:
   scheme_code       scheme_name  latest_nav        date
0       119551      SBI Bluechip   105.92190  19-06-2026
1       120503    ICICI Bluechip   107.96360  19-06-2026
2       118632  Nippon Large Cap   100.78240  19-06-2026
3       119092     Axis Bluechip  6195.78150  19-06-2026
4       120841    Kotak Bluechip   251.57130  19-06-2026


In [4]:
import pandas as pd

# Read fund master CSV
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

print("========== Unique Fund Houses ==========")
print(fund_master["fund_house"].unique())

print("\n========== Unique Categories ==========")
print(fund_master["category"].unique())

print("\n========== Unique Sub Categories ==========")
print(fund_master["sub_category"].unique())

print("\n========== Unique Risk Categories ==========")
print(fund_master["risk_category"].unique())

print("\n========== Total Records ==========")
print(fund_master.shape)

========== Unique Fund Houses ==========
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

========== Unique Categories ==========
['Equity' 'Debt']

========== Unique Sub Categories ==========
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

========== Unique Risk Categories ==========
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']

========== Total Records ==========
(40, 15)


In [5]:
import pandas as pd

# Load CSV
trans = pd.read_csv("../data/raw/08_investor_transactions.csv")

# Date format
trans["transaction_date"] = pd.to_datetime(trans["transaction_date"])

# Standardize transaction type
trans["transaction_type"] = trans["transaction_type"].str.upper()

# Amount > 0
trans = trans[trans["amount_inr"] > 0]

# Remove duplicates
trans = trans.drop_duplicates()

# KYC values standard
trans["kyc_status"] = trans["kyc_status"].str.capitalize()

# Save cleaned file
trans.to_csv("../data/processed/08_investor_transactions_clean.csv", index=False)

print(trans.head())
print(trans.shape)
print("Investor Transactions Cleaned Successfully!")

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       REDEMPTION      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          LUMPSUM        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [6]:
import pandas as pd

# Load CSV
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

# Return columns ko numeric banao
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct"]

for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")

# Expense ratio numeric
perf["expense_ratio_pct"] = pd.to_numeric(
    perf["expense_ratio_pct"], errors="coerce"
)

# Expense ratio 0.1% - 2.5% ke bahar ho to NaN
perf.loc[
    (perf["expense_ratio_pct"] < 0.1)
    | (perf["expense_ratio_pct"] > 2.5),
    "expense_ratio_pct"
] = pd.NA

# Duplicate rows hatao
perf = perf.drop_duplicates()

# Output dekho
print(perf.head())
print(perf.shape)
print("Scheme Performance Cleaned Successfully!")

   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0              11.49

In [7]:
from sqlalchemy import create_engine

# SQLite database create karo
engine = create_engine("sqlite:///bluestock_mf.db")

# CSV load karo
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
investor_transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

# SQLite me save karo
fund_master.to_sql("dim_fund", engine, if_exists="replace", index=False)
nav_history.to_sql("fact_nav", engine, if_exists="replace", index=False)
scheme_performance.to_sql("fact_performance", engine, if_exists="replace", index=False)
investor_transactions.to_sql("fact_transactions", engine, if_exists="replace", index=False)
aum.to_sql("fact_aum", engine, if_exists="replace", index=False)

print("SQLite Database Loaded Successfully!")
print("Tables Created:")
print("- dim_fund")
print("- fact_nav")
print("- fact_performance")
print("- fact_transactions")
print("- fact_aum")

SQLite Database Loaded Successfully!
Tables Created:
- dim_fund
- fact_nav
- fact_performance
- fact_transactions
- fact_aum


In [5]:
print(fund_master.columns.tolist())

['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']


In [6]:
fund_master = dataframes["01_fund_master"]

print("Unique Fund Houses:")
print(fund_master["fund_house"].unique())

print("\nUnique Categories:")
print(fund_master["category"].unique())

print("\nUnique Sub Categories:")
print(fund_master["sub_category"].unique())

print("\nUnique Risk Categories:")
print(fund_master["risk_category"].unique())

Unique Fund Houses:
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

Unique Categories:
['Equity' 'Debt']

Unique Sub Categories:
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

Unique Risk Categories:
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [7]:
fund_master = dataframes["01_fund_master"]
nav_history = dataframes["02_nav_history"]

master_codes = set(fund_master["amfi_code"])
nav_codes = set(nav_history["amfi_code"])

missing_codes = master_codes - nav_codes

print("Total Fund Master Codes:", len(master_codes))
print("Total NAV History Codes:", len(nav_codes))
print("Missing Codes:", len(missing_codes))

if len(missing_codes) == 0:
    print("✅ All AMFI codes are present in NAV History.")
else:
    print("❌ Missing AMFI Codes:")
    print(missing_codes)

Total Fund Master Codes: 40
Total NAV History Codes: 40
Missing Codes: 0
✅ All AMFI codes are present in NAV History.


In [8]:
for name, df in dataframes.items():

    print("=" * 60)
    print("Dataset:", name)

    print("\nShape:")
    print(df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\n")

Dataset: 01_fund_master

Shape:
(40, 15)

Missing Values:
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

Data Types:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

Duplicate Rows:
0


Dataset: 02_nav_history

Shape:
(46000, 3)

Missing Values:
amfi_code  